In [ ]:
import numpy as np
import pandas as pd
from transformers import AutoImageProcessor, AutoModel
import torch
from torch.utils.data import Dataset, Subset
import torch.nn as nn
import torch.nn.functional as F
import os
from pathlib import Path
from tqdm.auto import tqdm
import matplotlib.pyplot as plt
from PIL import Image
from sklearn.metrics.pairwise import cosine_similarity
import cv2
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import GroupKFold
from dataclasses import dataclass, asdict
from collections import defaultdict
from torch.utils.data import DataLoader
import random
from clearml import Task
from datetime import datetime

In [ ]:
@dataclass
class Config:
    device: str = "cuda"
    gkf_n_splits: int = 5
    n_epochs: int = 20

    hidden_dim: int = 512
    batch_size: int = 64
    learning_rate: float = 0.001
    momentum: float = 0.9

    # task
    model_name: str = "dino_mlp"
    task_version_name: str = "baseline"

    seed: int = 42


cfg = Config()

In [ ]:
# device = torch.device('cuda' if torch.cuda.is_available() else "cpu")
print("device:", cfg.device)

In [ ]:
def set_seed(seed):
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True
    torch.use_deterministic_algorithms(True, warn_only=True)


set_seed(cfg.seed)

In [ ]:
# task_name = f'{cfg.model_name}-{cfg.task_version_name}'
# task = Task.init(project_name='autonomous-driving/bc', task_name=task_name)
# task.connect(asdict(cfg))

In [ ]:
if Path("/kaggle/input").exists():
    pass
else:
    notebook_path = Path.cwd()
    root_path = notebook_path.parent.parent
    model_path = root_path / "model"
    comma_dataset_path = root_path / "data" / "comma2k19"
    comma_embeddings_path = root_path / "data" / "comma_dino_embeddings_1"
print(f"Comma path: {comma_dataset_path}")

# Building index

```
Dataset_chunk_n
|
+-- route_id (dongle_id|start_time)
    |
    +-- segment_number
        |
        +-- preview.png (first frame video)
        +-- raw_log.bz2 (raw capnp log, can be read with openpilot-tools: logreader)
        +-- video.hevc (video file, can be read with openpilot-tools: framereader)
        +-- processed_log/ (processed logs as numpy arrays, see format for details)
        +-- global_pos/ (global poses of camera as numpy arrays, see format for details)
```

In [ ]:
def build_index(comma_dataset_path):
    chunks = [x.name for x in Path(comma_dataset_path).iterdir() if x.is_dir()]
    samples = []
    for chunk in chunks:
        chunk_path = comma_dataset_path / chunk
        routes = [x for x in Path(chunk_path).iterdir() if x.is_dir()]
        for route in routes:
            segments = [x for x in Path(route).iterdir() if x.is_dir()]
            for segment in segments:
                video_path = segment / "video.hevc"

                steering_angle_path = (
                    segment / "processed_log" / "CAN" / "steering_angle"
                )
                steering_angle_value = np.load(steering_angle_path / "value")
                steering_angle_t = np.load(steering_angle_path / "t")

                frame_times_path = segment / "global_pose" / "frame_times"
                frame_times = np.load(frame_times_path)

                steering_angle_idx = np.searchsorted(steering_angle_t, frame_times)
                for i, frame_time in enumerate(frame_times):
                    idx = steering_angle_idx[i]
                    if idx == 0:
                        nearest = 0
                    elif idx == len(steering_angle_t):
                        nearest = len(steering_angle_t) - 1
                    else:
                        left = idx - 1
                        right = idx

                        if abs(frame_time - steering_angle_t[left]) < abs(
                            frame_time - steering_angle_t[right]
                        ):
                            nearest = left
                        else:
                            nearest = right

                    sample = {
                        "video_path": video_path,
                        "chunk": chunk,
                        "route": route.name,
                        "segment": segment.name,
                        "frame_idx": i,
                        "frame_time": frame_time,
                        "steering_angle": steering_angle_value[nearest],
                    }
                    samples.append(sample)
    return samples


samples = build_index(comma_dataset_path=comma_dataset_path)
samples[1]

In [ ]:
df = pd.DataFrame(samples)
df.head()

In [ ]:
df.steering_angle.plot()

In [ ]:
from IPython.display import Video
import cv2
import math


def draw_steering(frame, steering_angle):
    frame = frame.copy()

    h, w = frame.shape[:2]

    text = f"steering: {steering_angle:.2f} deg"
    cv2.putText(
        frame,
        text,
        (30, 50),
        cv2.FONT_HERSHEY_SIMPLEX,
        1.2,
        (0, 255, 255),
        2,
        cv2.LINE_AA,
    )

    center = (w // 2, h - 90)
    radius = 85
    angle_rad = -math.radians(float(steering_angle))

    cv2.circle(frame, center, radius, (255, 255, 255), 3)
    cv2.circle(frame, center, 8, (0, 255, 255), -1)

    spoke_angles = [-90, 30, 150]

    for spoke_angle in spoke_angles:
        a = math.radians(spoke_angle) + angle_rad

        end_x = int(center[0] + radius * 0.75 * math.cos(a))
        end_y = int(center[1] + radius * 0.75 * math.sin(a))

        cv2.line(
            frame,
            center,
            (end_x, end_y),
            (0, 255, 255),
            5,
            cv2.LINE_AA,
        )

    marker_angle = math.radians(-90) + angle_rad
    marker_x = int(center[0] + radius * math.cos(marker_angle))
    marker_y = int(center[1] + radius * math.sin(marker_angle))

    cv2.circle(frame, (marker_x, marker_y), 8, (0, 0, 255), -1)

    return frame


def make_steering_clip(df, output_path="steering_clip.mp4", fps=20):
    df = df.copy()
    df["frame_idx"] = df["frame_idx"].astype(int)
    df = df.sort_values(["video_path", "frame_idx"])

    writer = None

    for video_path, video_df in df.groupby("video_path", sort=False):
        cap = cv2.VideoCapture(str(video_path))

        if not cap.isOpened():
            print(f"Cannot open video: {video_path}")
            continue

        wanted = {int(row.frame_idx): row for row in video_df.itertuples(index=False)}

        max_frame_idx = max(wanted)

        current_idx = 0

        while current_idx <= max_frame_idx:
            ok, frame = cap.read()

            if not ok:
                break

            if current_idx in wanted:
                row = wanted[current_idx]

                frame = draw_steering(
                    frame,
                    steering_angle=float(row.steering_angle),
                )

                if writer is None:
                    h, w = frame.shape[:2]
                    writer = cv2.VideoWriter(
                        output_path,
                        cv2.VideoWriter_fourcc(*"mp4v"),
                        fps,
                        (w, h),
                    )
                    if not writer.isOpened():
                        raise RuntimeError(f"Cannot open video writer:{output_path}")

                writer.write(frame)

            current_idx += 1
        else:
            print("finished")

        cap.release()

    if writer is not None:
        writer.release()
    else:
        raise RuntimeError("No frames were written")

    return output_path


output_path = (
    "/home/hxastur/vscode-projects/autonomous-driving/data/created_video/steering.mp4"
)
selected_df = df[20000:30000]
clip_path = make_steering_clip(
    selected_df,
    output_path=output_path,
    fps=20,
)

In [ ]:
import subprocess
from IPython.display import Video

converted_path = output_path.replace(".mp4", "_browser.mp4")

subprocess.run(
    [
        "ffmpeg",
        "-y",
        "-i",
        output_path,
        "-vcodec",
        "libx264",
        "-pix_fmt",
        "yuv420p",
        "-movflags",
        "+faststart",
        converted_path,
    ],
    check=True,
)

Video(converted_path, embed=True)